# Quantra — Train on Colab

One universal **PPO** policy built to **repeatedly pass FTMO-style challenges** (2.5% daily target / 4% trailing wall) — not to maximize PnL.

An episode is **N trading days on ONE continuous account** (C10): the balance carries forward day to day, a day that hits the wall is **locked out for the rest of that day** and resets at midnight (it does **not** end the episode), and the episode ends only when the N days are exhausted or the account is blown. A **large end-of-day penalty** (C11) fires whenever the bot misses that day's target, so it learns **consistency**, not just survival.

> ⚠️ **Pick a CPU runtime** (Runtime → Change runtime type → CPU, or High-RAM). The locked net is a tiny **3×256 MLP (~207 inputs)** — the bottleneck is CPU env-stepping, not the GPU. The optimizer below will *prove* whether a GPU is worth it (it usually isn't), so you don't waste paid GPU hours.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git quantra_repo
%cd quantra_repo

## 2. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn, matplotlib. We only add the extras.

In [ ]:
!pip install -q gdown pyarrow psutil optuna seaborn statsmodels gymnasium tqdm
# nvidia-ml-py is only needed if you chose a GPU runtime:
import torch
if torch.cuda.is_available():
    !pip install -q nvidia-ml-py

## 3. Mount Google Drive (price data)
Your 4 symbols (MT5 1m, ~5yr) live in the `rl-trading-data` folder. Mounting lets the data loader read them without re-downloading 500 MB each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Expected path: /content/drive/MyDrive/rl-trading-data/<SYMBOL>_M1_*.csv
!ls -la "/content/drive/MyDrive/rl-trading-data" 2>/dev/null || echo "Adjust the folder path if your Drive layout differs; gdown-by-ID fallback is built into the loader."

## 4. Race the hardware (CPU vs GPU) and size to ~80%
This is the money-saver: it benchmarks both devices on the real four-head workload, picks the faster one (preferring CPU on a near-tie), scales parallel envs to ~80% utilisation, and reports what it measured.

In [ ]:
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs

ensure_dirs()
cfg = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as mon:
    hw = plan(state_dim=cfg.nominal_state_dim, hw=cfg.hardware)
util = mon.stop()

print_report(hw)
print('Utilisation during race:', util.render())

## 5. HYPERPARAMETERS — all visible + editable BEFORE any run
Every knob the training run uses is surfaced here. **🔴 = LOCKED** (visible but read-only; changing needs Monty's sign-off). Edit the values, then build the env/trainer from the objects this cell constructs — nothing downstream is hardcoded.

In [ ]:
# ── HYPERPARAMETERS (edit here; gamma/lambda are 🔴 LOCKED — visible only) ──
from quantra.runtime import config as qcfg
from quantra.learning_system.trainer.gae import GAMMA, LAMBDA            # 🔴 0.997 / 0.97
from quantra.learning_system.trainer.trainer import TrainConfig
from quantra.learning_system.trainer.scheduler import AggressionRanges

# --- Episode length (C10: N trading days on ONE continuous account) ----------
TRAINING_DAYS = qcfg.TRAINING_DAYS     # default 180 (~6 months) for a full training run
EVAL_DAYS     = qcfg.EVAL_DAYS         # default 8 for a Barbershop/eval window
EPISODE_DAYS  = TRAINING_DAYS          # what THIS run uses (swap to EVAL_DAYS for a short eval)

# --- Challenge numbers (what "passing" means; → make_challenge) --------------
DAILY_TARGET_PCT   = 2.5               # daily profit goal (% of that day's OPENING balance)
DAILY_RISK_PCT     = 4.0               # daily trailing-wall risk
FTMO_MODE          = True              # ON: 2-phase auto-flat at target. OFF: target is the aim
STOP_FOR_DAY       = False             # OFF-mode: bank + lock out the day at target (else run on)
FAILED_DAY_PENALTY = 5.0               # C11 weight — LARGE end-of-day penalty, ∝ shortfall from target
PERMANENT_DD_PCT   = 10.0              # -10% max wall — OBSERVATION only (C12), NOT enforced

# --- Reward weights (C16: plain-English, operator-tunable; math/structure unchanged) ---
# Layer 0 net-PnL is the dominant outcome base E8 protects (keep net_pnl_weight = 1.0). The four
# shaping weights are small "whispers". daily_progress_weight is the consistency driver (matters most).
NET_PNL_WEIGHT         = 1.0           # 🔴 keep 1.0 (E8 Layer-0 dominance)
STEP_PNL_WEIGHT        = 1e-4          # small per-bar in-position bonus
DAILY_PROGRESS_WEIGHT  = 1e-3          # the consistency driver (raised from 1e-4)
DRAWDOWN_PAIN_WEIGHT   = 5e-4          # pain-zone penalty near the wall
DRAWDOWN_PAIN_STEEPNESS= 4.0           # exponential steepness of the pain ramp
TRADE_QUALITY_WEIGHT   = 5e-5          # close-quality / target-progress whisper

# --- Enforcement / safety toggles -------------------------------------------
TRAINING_WHEELS = qcfg.TRAINING_WHEELS  # counter-trend OPEN blocks (True = on)
TRAINING_PHASE  = qcfg.TRAINING_PHASE   # PHASE_FREE (signals observation-only) / PHASE_CONSTRAINED

# --- PPO discount (🔴 LOCKED — shown for visibility, do not change) ----------
GAMMA_LOCKED  = GAMMA                  # 🔴 0.997  (the "math of patience")
LAMBDA_LOCKED = LAMBDA                 # 🔴 0.97

# --- Aggression dial RANGES (the missed-opportunity scheduler picks within these) ---
ENTROPY_RANGE = (0.03, 0.08)
CLIP_RANGE    = (0.25, 0.35)
LR_RANGE      = (5e-4, 1e-3)
EPOCHS_RANGE  = (10, 15)

# --- Rollout / optimization --------------------------------------------------
ROLLOUT_SIZE   = 512
MINIBATCH_SIZE = 64
VALUE_COEF     = 0.5

# Build the REAL config objects the env + Trainer consume (nothing hardcoded downstream).
CHALLENGE  = qcfg.make_challenge(daily_target_pct=DAILY_TARGET_PCT, daily_risk_pct=DAILY_RISK_PCT,
                                 ftmo_mode=FTMO_MODE, stop_for_day=STOP_FOR_DAY,
                                 permanent_dd_pct=PERMANENT_DD_PCT,
                                 failed_day_penalty=FAILED_DAY_PENALTY)
REWARD_CFG = qcfg.RewardConfig(net_pnl_weight=NET_PNL_WEIGHT, step_pnl_weight=STEP_PNL_WEIGHT,
                               daily_progress_weight=DAILY_PROGRESS_WEIGHT,
                               drawdown_pain_weight=DRAWDOWN_PAIN_WEIGHT,
                               drawdown_pain_steepness=DRAWDOWN_PAIN_STEEPNESS,
                               trade_quality_weight=TRADE_QUALITY_WEIGHT,
                               failed_day_penalty=FAILED_DAY_PENALTY)
TRAIN_CFG  = TrainConfig(rollout_size=ROLLOUT_SIZE, minibatch=MINIBATCH_SIZE, value_coef=VALUE_COEF)
AGG_RANGES = AggressionRanges(entropy=ENTROPY_RANGE, clip=CLIP_RANGE, lr=LR_RANGE, epochs=EPOCHS_RANGE)
# (the env is built as TradingEnv(..., challenge=CHALLENGE, reward_cfg=REWARD_CFG, episode_days=EPISODE_DAYS))

# --- C19: OVERRIDES = ONLY the knobs that differ from baseline -> auto-names + reproduces the policy ---
# build_overrides_dict diffs the live config objects above against the dataclass defaults. The result
# NAMES the policy (auto_name) and is saved verbatim in its manifest by the Barbershop loop (Cell 6),
# so the Leaderboard can tell policies apart and every run is reproducible. Edit knobs ABOVE, not here.
from quantra.learning_system.policy_registry import auto_name
OVERRIDES = qcfg.build_overrides_dict(challenge=CHALLENGE, reward=REWARD_CFG, train=TRAIN_CFG,
                                      training_phase=TRAINING_PHASE, training_wheels=TRAINING_WHEELS)
POLICY_NAME, _name_basis = auto_name(OVERRIDES)   # base_policy is supplied by RESUME_FROM in Barbershop

print(f"Episode: {EPISODE_DAYS} days/episode (one continuous account) | failed-day penalty={FAILED_DAY_PENALTY}")
print(f"Challenge: +{CHALLENGE.daily_target_pct}%/day, {CHALLENGE.daily_risk_pct}% wall | gamma/lambda LOCKED {GAMMA_LOCKED}/{LAMBDA_LOCKED}")
print(f"Reward weights: net_pnl={NET_PNL_WEIGHT} step_pnl={STEP_PNL_WEIGHT} daily_progress={DAILY_PROGRESS_WEIGHT} pain={DRAWDOWN_PAIN_WEIGHT} trade_quality={TRADE_QUALITY_WEIGHT}")
print(f"Aggression ranges: entropy={ENTROPY_RANGE} clip={CLIP_RANGE} lr={LR_RANGE} epochs={EPOCHS_RANGE} | rollout={ROLLOUT_SIZE}/{MINIBATCH_SIZE}")
print(f"OVERRIDES (diff vs baseline): {OVERRIDES if OVERRIDES else '{}  (pure baseline)'}")
print(f"Auto policy name: {POLICY_NAME}")

## 5. Train
The training entry point lands as milestones **M1–M8** are built (data → features → laws → env → agent → reward → curriculum → trainer). It will be:
```python
python -m quantra.learning_system.trainer  # coming as M8 lands
```
Until then, the cell above already proves the repo runs and the optimizer selects the cheapest fast device for your runtime.